In [1]:
import pickle

with open("X_train_ngram.pkl", "rb") as f:
    X_train = pickle.load(f)

with open("X_test_ngram.pkl", "rb") as f:
    X_test = pickle.load(f)

with open("y_train_ngram.pkl", "rb") as f:
    y_train = pickle.load(f)

with open("y_test_ngram.pkl", "rb") as f:
    y_test = pickle.load(f)

with open("ngrams.pkl", "rb") as f:
    ngram = pickle.load(f)

In [2]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# Define parameter grid for each model
param_grid = {
    "LR": {
        "model": LogisticRegression(max_iter=1000, random_state=42),
        "params": {"C": [0.1, 0.5, 1, 2, 5]}
    },
    "SVM Linear": {
        "model": SVC(kernel="linear", random_state=42),
        "params": {"C": [0.1, 0.5, 1, 2, 5]}
    },
    "SVM RBF": {
        "model": SVC(kernel="rbf", random_state=42),
        "params": {"C": [0.5, 1, 2, 5], "gamma": ["scale", "auto", 0.1]}
    },
    "Multinomial NB": {
        "model": MultinomialNB(),
        "params": {"alpha": [0.1, 0.3, 0.5, 1.0, 2.0]}
    },
    "RF": {
        "model": RandomForestClassifier(random_state=42),
        "params": {
            "n_estimators": [100, 200, 300],
            "max_depth": [10, 20, None],
            "min_samples_split": [2, 5, 10]
        }
    }
}

print("MODEL SELECTION WITH PROPER 5-FOLD CROSS VALIDATION")

# STEP 1: Hyperparameter tuning
print("\nTuning hyperparameters with 5-fold CV on training data...")

results = []
best_models = {}

for model_name, model_params in param_grid.items():
    print("Tuning:", model_name)
    
    # Create Stratified K-Fold
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    # GridSearchCV
    grid = GridSearchCV(
        estimator=model_params["model"],
        param_grid=model_params["params"],
        cv=skf,
        scoring="accuracy",
        n_jobs=-1,
        verbose=0
    )
    
    # Fit on training data
    grid.fit(X_train, y_train)
    
    # Store best model
    best_models[model_name] = grid.best_estimator_
    
    # Store results
    results.append({
        "Model": model_name,
        "Best Params": grid.best_params_,
        "Train CV Accuracy": round(grid.best_score_, 4),
        "Best Estimator": grid.best_estimator_
    })
    
    print("Best CV Accuracy:", grid.best_score_)
    print("Best Params:", grid.best_params_)

# STEP 2: Select best model based on CV
df_results = pd.DataFrame(results)
df_results = df_results.sort_values(by="Train CV Accuracy", ascending=False)

print("MODEL RANKING (based on 5-fold CV)")
print(df_results[["Model", "Train CV Accuracy", "Best Params"]])

best_model_name = df_results.iloc[0]["Model"]
best_model = best_models[best_model_name]

print("\nSELECTED BEST MODEL:", best_model_name)
print("Best CV Score:", df_results.iloc[0]['Train CV Accuracy'])

# STEP 3: Retrain best model on full training data
print("RETRAINING BEST MODEL ON FULL TRAINING DATA")

best_model.fit(X_train, y_train)
print("Model retrained on full training data")

# STEP 4: Evaluate on test set
print("FINAL EVALUATION ON TEST SET (ONE TIME ONLY!)")

y_pred = best_model.predict(X_test)

test_accuracy = accuracy_score(y_test, y_pred)
test_f1 = f1_score(y_test, y_pred, average="binary")

print("\nFINAL TEST RESULTS:")
print("Test Accuracy:", test_accuracy)
print("Test F1 Score:", test_f1)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Fake', 'Real']))

cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(cm)

# STEP 5: Compare all models on test set
print("COMPARISON WITH ALL MODELS ON TEST SET")

comparison_results = []

for model_name, estimator in best_models.items():
    estimator.fit(X_train, y_train)
    y_pred_model = estimator.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred_model)
    f1 = f1_score(y_test, y_pred_model, average="binary")
    
    comparison_results.append({
        "Model": model_name,
        "Test Accuracy": acc,
        "Test F1": f1
    })

comparison_df = pd.DataFrame(comparison_results)
comparison_df = comparison_df.sort_values(by=["Test F1", "Test Accuracy"], ascending=False)

print("TEST SET PERFORMANCE OF ALL MODELS")
print(comparison_df)

best_test_model_name = comparison_df.iloc[0]["Model"]
best_test_f1_score = comparison_df.iloc[0]["Test F1"]
best_test_accuracy_score = comparison_df.iloc[0]["Test Accuracy"]

print("BEST MODEL BASED ON TEST SET PERFORMANCE")
print("Best Test Model:", best_test_model_name)
print("Test Accuracy:", best_test_accuracy_score)
print("Test F1 Score:", best_test_f1_score)

MODEL SELECTION WITH PROPER 5-FOLD CROSS VALIDATION

Tuning hyperparameters with 5-fold CV on training data...
Tuning: LR
Best CV Accuracy: 0.6008078079624787
Best Params: {'C': 0.1}
Tuning: SVM Linear
Best CV Accuracy: 0.5835559192726166
Best Params: {'C': 0.1}
Tuning: SVM RBF
Best CV Accuracy: 0.6068970801531935
Best Params: {'C': 1, 'gamma': 0.1}
Tuning: Multinomial NB
Best CV Accuracy: 0.5995683721308795
Best Params: {'alpha': 2.0}
Tuning: RF
Best CV Accuracy: 0.6059966260791492
Best Params: {'max_depth': None, 'min_samples_split': 10, 'n_estimators': 200}
MODEL RANKING (based on 5-fold CV)
            Model  Train CV Accuracy  \
2         SVM RBF             0.6069   
4              RF             0.6060   
0              LR             0.6008   
3  Multinomial NB             0.5996   
1      SVM Linear             0.5836   

                                         Best Params  
2                             {'C': 1, 'gamma': 0.1}  
4  {'max_depth': None, 'min_samples_split': 10,

In [3]:
import pickle

with open("best_ngram_liar.pkl", "wb") as f:
    pickle.dump(best_model, f)

print("Best model saved successfully")

Best model saved successfully
